# Supervised Learning - Logistic Regression

Fraud classifier trained on the original features built in `preprocessing.ipynb`.

Input: `data/processed/fraudTrain_features.csv`, `data/processed/fraudTest_features.csv`

This notebook covers:
1. Loading the processed data
2. Train / validation / test split (out-of-time)
3. Model pipeline (scaling + class-weighted logistic regression)
4. Validation: choosing `C` and the decision threshold
5. Test-set evaluation
6. Feature importance through coefficients and odds ratios
7. Results summary

**Metrics.** Fraud is highly imbalanced, so the main ranking metric is **PR-AUC** (average precision). Threshold-dependent results use the same standalone schema as the XGBoost and Random Forest notebooks: alerts, frauds caught, false alarms, precision, and recall.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
%matplotlib inline

DATA_DIR = Path("data/processed")

# Chart colours: blue/orange categorical, blue<->red diverging, muted greys for axes
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e5e1"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 12,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "figure.dpi": 110,
})

## 1. Load data

In [ ]:
train = pd.read_csv(DATA_DIR / "fraudTrain_features.csv")
test = pd.read_csv(DATA_DIR / "fraudTest_features.csv")

X_train_full = train.drop(columns=["is_fraud"]).astype("float32")
y_train_full = train["is_fraud"]
X_test = test.drop(columns=["is_fraud"]).astype("float32")
y_test = test["is_fraud"]

# Use only the original features; exclude optional engineered KYC/behavioural columns.
cols = [c for c in X_train_full.columns if not c.startswith(("kyc_", "beh_"))]
print(f"train: {len(X_train_full):,} rows, {len(cols):,} original features, fraud rate {y_train_full.mean():.3%}")
print(f"test:  {len(X_test):,} rows, fraud rate {y_test.mean():.3%}")


## 2. Train / validation / test split

The data is split three ways:

| Split | Rows | Period | Used for |
|---|---|---|---|
| **Train (fit)** | first 80% of `fraudTrain` | Jan 2019 – Mar 2020 | fitting model coefficients |
| **Validation** | last 20% of `fraudTrain` | Mar 2020 – Jun 2020 | choosing `C` and the decision threshold |
| **Test** | all of `fraudTest` | Jun 2020 – Dec 2020 | final evaluation, used once |

The training rows are in time order, so taking the last 20% as validation gives an **out-of-time** split, the same way the test set relates to the training set. A random split would let the model see transactions from the same days it is validated on, which overstates performance on future data.

In [ ]:
cut = int(len(X_train_full) * 0.8)
X_tr, y_tr = X_train_full.iloc[:cut], y_train_full.iloc[:cut]
X_val, y_val = X_train_full.iloc[cut:], y_train_full.iloc[cut:]

pd.DataFrame({
    "rows": [len(X_tr), len(X_val), len(X_test)],
    "fraud": [y_tr.sum(), y_val.sum(), y_test.sum()],
    "fraud rate": [f"{y_tr.mean():.3%}", f"{y_val.mean():.3%}", f"{y_test.mean():.3%}"],
}, index=["train (fit)", "validation", "test"])

## 3. Model pipeline

- **Scaling.** Continuous features are standardised so each coefficient means "change in log-odds per 1 standard deviation". The 0/1 columns are left as-is. The scaler is fit on the training rows only, inside the pipeline.
- **`class_weight="balanced"`.** Fraud cases receive higher weight so the model cannot ignore the minority class. Because this affects probability calibration, the decision threshold is tuned on validation data rather than fixed at 0.5.
- **Regularisation.** L2 penalty; its strength `C` is chosen on the validation set.


In [ ]:
binary_columns = [c for c in X_train_full.columns if X_train_full[c].isin([0, 1]).all()]
print(f"{len(cols) - len([c for c in binary_columns if c in cols])} continuous (scaled), {len([c for c in binary_columns if c in cols])} binary (left as 0/1)")


def make_model(C, columns):
    continuous = [c for c in columns if c not in binary_columns]
    return Pipeline([
        ("scale", ColumnTransformer(
            [("continuous", StandardScaler(), continuous)],
            remainder="passthrough",
            verbose_feature_names_out=False,
        )),
        ("logreg", LogisticRegression(C=C, class_weight="balanced", max_iter=3000)),
    ])


def best_f1_threshold(y, proba):
    precision, recall, thresholds = precision_recall_curve(y, proba)
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[np.argmax(f1)])


def scores_at(y, proba, threshold):
    alert = proba >= threshold
    caught = int((alert & (y == 1)).sum())
    return {
        "alerts": int(alert.sum()),
        "frauds caught": caught,
        "false alarms": int(alert.sum()) - caught,
        "precision": caught / max(int(alert.sum()), 1),
        "recall": caught / int((y == 1).sum()),
    }


## 4. Validation: choosing `C` and the threshold

Each candidate `C` is fit on the train rows and scored on validation by PR-AUC. The best `C` is kept, and the decision threshold is set to the value that maximises F1 on validation. Nothing in this section touches the test set.


In [ ]:
C_GRID = [0.00001, 0.0001, 0.001, 0.01, 0.1]

c_search, best = [], None
for C in C_GRID:
    model = make_model(C, cols).fit(X_tr[cols], y_tr)
    proba = model.predict_proba(X_val[cols])[:, 1]
    pr_auc = average_precision_score(y_val, proba)
    c_search.append({"C": C, "val PR-AUC": pr_auc})
    if best is None or pr_auc > best["val PR-AUC"]:
        best = {"C": C, "val PR-AUC": pr_auc, "val_proba": proba}

best["threshold"] = best_f1_threshold(y_val, best["val_proba"])
print(f"Best C: {best['C']}; standalone threshold: {best['threshold']:.3f}")
pd.DataFrame(c_search).round(4)


In [ ]:
pd.DataFrame({f"standalone ({best['threshold']:.3f})": scores_at(y_val, best["val_proba"], best["threshold"])}).T.round(3)


## 5. Test-set evaluation

With `C` and the threshold fixed from validation, the model is refit on the full training set and evaluated once on the test set.


In [ ]:
final_model = make_model(best["C"], cols).fit(X_train_full[cols], y_train_full)
test_proba = final_model.predict_proba(X_test[cols])[:, 1]
threshold = best["threshold"]

print(f"Test PR-AUC {average_precision_score(y_test, test_proba):.4f} | ROC-AUC {roc_auc_score(y_test, test_proba):.4f}")
pd.DataFrame({f"standalone ({threshold:.3f})": scores_at(y_test, test_proba, threshold)}).T.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
p, r, _ = precision_recall_curve(y_test, test_proba)
ax.plot(r[:-1], p[:-1], color=BLUE,
        label=f"Logistic regression (PR-AUC {average_precision_score(y_test, test_proba):.3f})")
ax.axhline(y_test.mean(), color=MUTED, linestyle="--", linewidth=1)
ax.annotate("random classifier", (0.02, y_test.mean()), xytext=(0, 5),
            textcoords="offset points", color=MUTED, fontsize=9)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_ylim(0, 1.02)
ax.set_title("Precision-recall on the test set", loc="left")
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()


## 6. Model details

The model uses the original processed feature set.


In [ ]:
val_proba = best["val_proba"]
threshold = best["threshold"]

precision, recall, thresholds = precision_recall_curve(y_val, val_proba)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precision[:-1], color=BLUE, label="Precision")
ax.plot(thresholds, recall[:-1], color=ORANGE, label="Recall")
ax.axvline(threshold, color=MUTED, linestyle="--", linewidth=1)
ax.annotate(f"chosen threshold = {threshold:.2f}", (threshold, 0.03),
            xytext=(-6, 0), textcoords="offset points", ha="right", color=INK)
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.02)
ax.set_title("Precision and recall vs threshold (validation)", loc="left")
ax.legend(frameon=False, loc="center left")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

p, r, _ = precision_recall_curve(y_test, test_proba)
axes[0].plot(r[:-1], p[:-1], color=BLUE)  # drop sklearn's artificial (recall=0, precision=1) end point
axes[0].axhline(y_test.mean(), color=MUTED, linestyle="--", linewidth=1)
axes[0].annotate("random classifier", (0.02, y_test.mean()), xytext=(0, 5),
                 textcoords="offset points", color=MUTED, fontsize=9)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_ylim(0, 1.02)
axes[0].set_title(f"Precision–recall (test), PR-AUC = {average_precision_score(y_test, test_proba):.3f}", loc="left")

fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[1].plot(fpr, tpr, color=BLUE)
axes[1].plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title(f"ROC (test), ROC-AUC = {roc_auc_score(y_test, test_proba):.3f}", loc="left")

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, (test_proba >= threshold).astype(int))
labels = ["legitimate", "fraud"]

fig, ax = plt.subplots(figsize=(4.8, 4))
ax.imshow(np.log10(cm + 1), cmap="Blues")
ax.grid(False)
for i in range(2):
    for j in range(2):
        share = cm[i, j] / cm[i].sum()
        ax.text(j, i, f"{cm[i, j]:,}\n({share:.1%} of row)", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 10 else INK)
ax.set_xticks([0, 1], labels)
ax.set_yticks([0, 1], labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix (test, threshold {threshold:.2f})", loc="left")
plt.tight_layout()
plt.show()

pd.DataFrame(cm, index=[f"actual {l}" for l in labels], columns=[f"predicted {l}" for l in labels])

## 7. Coefficients and odds ratios

Because multicollinearity was removed in preprocessing, the coefficients can be read individually:

- **Continuous features:** odds ratio = multiplicative change in the odds of fraud for a **1 standard deviation** increase.
- **Dummies:** odds ratio compares the feature to its baseline category.

Odds ratio > 1 means higher fraud risk, < 1 lower. These are associations within simulated data, not causal effects.


In [ ]:
feature_names = final_model.named_steps["scale"].get_feature_names_out()
coefs = pd.DataFrame({
    "feature": feature_names,
    "coef": final_model.named_steps["logreg"].coef_[0],
})
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs["scaled"] = ~coefs["feature"].isin(binary_columns)
coefs = coefs.reindex(coefs["coef"].abs().sort_values(ascending=False).index).reset_index(drop=True)
coefs.head(20).round(3)

In [ ]:
top = coefs.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(top["feature"], top["coef"], color=np.where(top["coef"] > 0, RED, BLUE), height=0.7)
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_axisbelow(True)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Coefficient (log-odds)  ← lower fraud risk | higher fraud risk →")
ax.set_title("Top 20 coefficients (full model)", loc="left")
plt.tight_layout()
plt.show()

## 8. Results summary

The notebook now reports one logistic-regression model trained on the original processed feature set. Compare its test PR-AUC, ROC-AUC, and standalone threshold metrics directly with the corresponding outputs from `xgboost.ipynb` and `random_forest.ipynb`.

All hyperparameters and the decision threshold are selected using the validation period only. The test period is reserved for the final evaluation.
